# Sistema RAG para normativa de la Universidad de El Salvador

**Notebook de demostración para la defensa técnica.** Presenta el pipeline final de forma breve y trazable; no sustituye a `LaboratorioML.ipynb`, que conserva el desarrollo experimental completo.

> Las celdas se entregan sin outputs simulados. Para ejecutarlas se necesitan las dependencias de `requirements.txt` y acceso local a los modelos descargados desde Hugging Face.

## 1. Objetivo de la demostración

Mostrar, paso a paso, cómo el sistema carga normativa institucional, conserva la procedencia de cada fragmento, recupera evidencia mediante MPNet y FAISS, y entrega esa evidencia a BETO-SQAC para extraer una respuesta.

## 2. ¿Qué problema resuelve el sistema?

Un PDF completo puede superar el contexto útil del modelo QA. El sistema divide el corpus en fragmentos y usa búsqueda semántica para seleccionar únicamente los diez candidatos más relacionados con la pregunta. Así reduce el espacio sometido a QA y conserva documento, página y chunk como fuente.

En este laboratorio, RAG no termina en un LLM generativo: BETO-SQAC realiza **Question Answering extractivo** y selecciona un span que ya existe en el texto recuperado.

## 3. Arquitectura general

### Fase de indexación

```text
PDF → extracción → limpieza → chunking → fragmentos + metadata
    → MPNet → embeddings normalizados → FAISS IndexFlatIP
```

### Fase de consulta

```text
Pregunta → MPNet → embedding de la pregunta → FAISS
         → Top-K chunks → BETO-SQAC → candidatos
         → selección por score QA → respuesta + fuente
```

La similitud de recuperación decide qué fragmentos llegan a QA. El score QA decide qué span candidato se presenta como respuesta. Son valores distintos.

## 4. Configuración final

| Componente | Configuración validada |
|---|---|
| Modelo QA | `MMG/bert-base-spanish-wwm-cased-finetuned-sqac` |
| Embeddings | `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` |
| Chunk size | 800 caracteres |
| Overlap | 0 |
| Top-K | 10 |
| Índice | FAISS `IndexFlatIP` |

Estos valores provienen del laboratorio ejecutado; la demo no vuelve a optimizarlos.

In [1]:
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found.*")

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from IPython.utils.capture import capture_output
from sentence_transformers import SentenceTransformer

from src.funciones_qa import (
    ExtractorQAManual,
    buscar_chunks_multidocumento,
    construir_indice_faiss_multidocumento,
    crear_chunks_multidocumento,
    extraer_documento_pdf,
    rag_multidocumento,
    validar_metadata_chunks,
)

MODELO_QA = "MMG/bert-base-spanish-wwm-cased-finetuned-sqac"
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
CHUNK_SIZE = 800
OVERLAP = 0
TOP_K = 10

In [2]:
configuracion = pd.DataFrame(
    {
        "Valor": [
            MODELO_QA,
            MODELO_EMBEDDINGS,
            CHUNK_SIZE,
            OVERLAP,
            TOP_K,
            "IndexFlatIP",
        ]
    },
    index=["QA", "Embeddings", "Chunk size", "Overlap", "Top-K", "FAISS"],
)
display(configuracion)

,Valor
QA,MMG/bert-base-spanish-wwm-cased-finetuned-sqac
Embeddings,sentence-transformers/paraphrase-multilingual-...
Chunk size,800
Overlap,0
Top-K,10
FAISS,IndexFlatIP


## 5. Carga del corpus

La primera demostración utiliza únicamente `data/documento_fuente.pdf`. Las rutas se resuelven desde la raíz del repositorio para mantener portabilidad.

In [3]:
RAIZ = Path.cwd().resolve()
PDF_PRINCIPAL = RAIZ / "data" / "documento_fuente.pdf"
PREGUNTAS = RAIZ / "data" / "preguntas_multidocumento.json"

if not (RAIZ / "src" / "funciones_qa.py").is_file():
    raise RuntimeError("Abra el notebook desde la raíz del repositorio")
if not PDF_PRINCIPAL.is_file():
    raise FileNotFoundError(PDF_PRINCIPAL)

documento_mono = extraer_documento_pdf(PDF_PRINCIPAL, documento_id=1)
print(f"Documento cargado: {documento_mono['documento']}")

Documento cargado: documento_fuente.pdf


In [4]:
resumen_documento = pd.DataFrame(
    [
        {
            "Documento": documento_mono["documento"],
            "Páginas": documento_mono["numero_paginas"],
            "Páginas con texto": documento_mono["paginas_con_texto"],
            "Caracteres": documento_mono["caracteres"],
            "Palabras aproximadas": documento_mono["palabras_aproximadas"],
        }
    ]
)
display(resumen_documento)

,Documento,Páginas,Páginas con texto,Caracteres,Palabras aproximadas
0,documento_fuente.pdf,62,62,173419,26119


## 6. Extracción y preparación del texto

`extraer_documento_pdf` usa PyMuPDF, procesa cada página por separado y aplica una limpieza conservadora. Además del texto unido, conserva los offsets inicial y final de cada página. Esos offsets permiten asociar después un chunk con su página física.

In [5]:
pagina_inicial = documento_mono["paginas"][0]
display(
    pd.DataFrame(
        [
            {
                "Página": pagina_inicial["pagina"],
                "Inicio": pagina_inicial["inicio"],
                "Fin": pagina_inicial["fin"],
                "Tiene texto": pagina_inicial["tiene_texto"],
                "Muestra": pagina_inicial["texto"][:240].replace("\n", " ") + "…",
            }
        ]
    )
)

,Página,Inicio,Fin,Tiene texto,Muestra
0,1,0,2210,True,1 ACUERDO N° 106/2011-2013 (V) LA ASAMBLEA GEN...


## 7. Fragmentación del documento

Se crean ventanas consecutivas de 800 caracteres y overlap cero. La función reutilizada acepta una lista de documentos; una lista de un elemento representa el caso monodocumento.

In [6]:
metadata_mono = crear_chunks_multidocumento(
    [documento_mono],
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
)

validacion_mono = validar_metadata_chunks(metadata_mono)
assert len(metadata_mono) == 217
print(f"Chunks creados: {len(metadata_mono)}")
print(f"Metadata alineada para {validacion_mono['chunks']} chunks")

Chunks creados: 217
Metadata alineada para 217 chunks


In [7]:
longitudes = np.array([chunk["num_caracteres"] for chunk in metadata_mono])
display(
    pd.DataFrame(
        [
            {
                "Chunks": len(metadata_mono),
                "Longitud mínima": int(longitudes.min()),
                "Longitud máxima": int(longitudes.max()),
                "Longitud promedio": float(longitudes.mean()),
            }
        ]
    ).style.format({"Longitud promedio": "{:.1f}"})
)

,Chunks,Longitud mínima,Longitud máxima,Longitud promedio
0,217,619,800,799.2


## 8. Metadata de los fragmentos

Cada vector tiene una fila de metadata con documento, página, chunk y posición vectorial. La igualdad `vector_posicion == posición en FAISS` es la base de la trazabilidad.

In [8]:
tabla_metadata = pd.DataFrame(
    [
        {
            "Documento": chunk["documento"],
            "Página inicial": chunk["pagina_inicio"],
            "Página final": chunk["pagina_fin"],
            "Chunk": chunk["chunk_id"],
            "Posición FAISS": chunk["vector_posicion"],
            "Texto": chunk["texto"][:180].replace("\n", " ") + "…",
        }
        for chunk in metadata_mono[:5]
    ]
)
display(tabla_metadata)

,Documento,Página inicial,Página final,Chunk,Posición FAISS,Texto
0,documento_fuente.pdf,1,1,1,0,1 ACUERDO N° 106/2011-2013 (V) LA ASAMBLEA GEN...
1,documento_fuente.pdf,1,1,2,1,ha materia. IV. Que de conformidad a los artíc...
2,documento_fuente.pdf,1,2,3,2,ento es normar y desarrollar las disposiciones...
3,documento_fuente.pdf,2,2,4,3,drá ser contrariado por normas contenidas en o...
4,documento_fuente.pdf,2,2,5,4,Asuntos Académicos. d) Comité de Ingreso Unive...


## 9. Generación de embeddings

MPNet transforma cada texto en un vector de 768 dimensiones. Los vectores se normalizan a norma L2 igual a uno; por eso el producto interno usado por FAISS produce el mismo ranking que la similitud coseno.

En esta celda también se carga una sola vez el pipeline BETO-SQAC que se utilizará después.

In [9]:
dispositivo_modelos = "cuda" if torch.cuda.is_available() else "cpu"

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDINGS,
    device=dispositivo_modelos,
)
pipeline_qa = ExtractorQAManual(MODELO_QA, dispositivo_modelos)

print(f"Embeddings en: {dispositivo_modelos}")
print(f"QA en: {dispositivo_modelos}")

Embeddings en: cuda
QA en: cuda


In [10]:
with capture_output():
    embeddings_mono, indice_mono, estadisticas_mono = (
        construir_indice_faiss_multidocumento(
            modelo_embeddings, metadata_mono, batch_size=32
        )
    )

normas = np.linalg.norm(embeddings_mono, axis=1)
assert embeddings_mono.shape == (217, 768)
assert np.allclose(normas, 1.0, atol=1e-5)

In [11]:
display(
    pd.DataFrame(
        [
            {
                "Modelo": MODELO_EMBEDDINGS,
                "Matriz": str(embeddings_mono.shape),
                "Dimensión": embeddings_mono.shape[1],
                "Norma mínima": float(normas.min()),
                "Norma máxima": float(normas.max()),
            }
        ]
    ).style.format({"Norma mínima": "{:.6f}", "Norma máxima": "{:.6f}"})
)

,Modelo,Matriz,Dimensión,Norma mínima,Norma máxima
0,sentence-transformers/paraphrase-multilingual-mpnet-base-v2,"(217, 768)",768,1.000000,1.000000


## 10. Construcción del índice FAISS

`IndexFlatIP` almacena todos los vectores y realiza una búsqueda exacta por producto interno. No entrena centroides ni aproxima vecinos. La correspondencia entre posición vectorial y metadata se valida antes de consultar.

In [12]:
assert indice_mono.ntotal == len(metadata_mono)
assert indice_mono.d == embeddings_mono.shape[1]
display(pd.DataFrame([estadisticas_mono]))

,tipo_indice,numero_vectores,dimension,tiempo_embeddings_s,tiempo_indice_s,memoria_embeddings_bytes
0,IndexFlatIP,217,768,1.738597,0.000095,666624


## 11. ¿Qué ocurre cuando llega una pregunta?

1. Se valida que la pregunta no esté vacía.
2. MPNet crea y normaliza su embedding.
3. FAISS devuelve los diez vectores más similares.
4. La metadata recupera documento, página, chunk y texto.
5. BETO-SQAC analiza cada uno de los diez textos.
6. Se selecciona el candidato con mayor **score QA**.
7. La respuesta se presenta con su fuente.

## 12. Embedding de la pregunta

La pregunta de ejemplo se toma del dataset existente; no se crea una etiqueta nueva.

In [13]:
preguntas_existentes = json.loads(PREGUNTAS.read_text(encoding="utf-8"))
item_demo_mono = next(
    item
    for item in preguntas_existentes
    if item["documento_esperado"] == PDF_PRINCIPAL.name
    and not item["sin_respuesta"]
)
pregunta_demo = item_demo_mono["pregunta"]

print("Pregunta:")
print(pregunta_demo)

Pregunta:
¿Con qué periodicidad sesiona ordinariamente el Consejo Académico?


In [14]:
embedding_pregunta = modelo_embeddings.encode(
    [pregunta_demo],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)

display(
    pd.DataFrame(
        [
            {
                "Forma": str(embedding_pregunta.shape),
                "Dimensión": embedding_pregunta.shape[1],
                "Norma L2": float(np.linalg.norm(embedding_pregunta[0])),
            }
        ]
    ).style.format({"Norma L2": "{:.6f}"})
)

,Forma,Dimensión,Norma L2
0,"(1, 768)",768,1.000000


## 13. Recuperación semántica Top-K

La recuperación usa exclusivamente el embedding de la pregunta. En esta etapa todavía no se ha ejecutado BETO-SQAC.

In [15]:
recuperados_mono = buscar_chunks_multidocumento(
    pregunta_demo,
    modelo_embeddings,
    indice_mono,
    metadata_mono,
    top_k=TOP_K,
)
assert len(recuperados_mono) == TOP_K

## 14. Visualización de fragmentos recuperados

La tabla separa el rank y la similitud de recuperación de cualquier score producido posteriormente por QA.

In [16]:
def tabla_recuperacion(recuperados):
    """Convierte los candidatos recuperados en una tabla para la defensa."""
    return pd.DataFrame(
        [
            {
                "Rank": chunk["rank"],
                "Documento": chunk["documento"],
                "Página": chunk["pagina"],
                "Chunk": chunk["chunk_id"],
                "Similitud de recuperación": chunk["similitud"],
                "Muestra": chunk["texto"][:150].replace("\n", " ") + "…",
            }
            for chunk in recuperados
        ]
    )

display(
    tabla_recuperacion(recuperados_mono).style.format(
        {"Similitud de recuperación": "{:.6f}"}
    )
)

,Rank,Documento,Página,Chunk,Similitud de recuperación,Muestra
0,1,documento_fuente.pdf,36,125,0.682253,s administraciones Académicas Locales enviaran a la Administración Académica Central los colectores de las evaluaciones realizadas por los estudiantes…
1,2,documento_fuente.pdf,2,6,0.639237,dor y tendrá las facultades que le otorga la legislación universitaria en general y en particular el presente Reglamento. CONSEJO ACADÉMICO. De la int…
2,3,documento_fuente.pdf,12,44,0.637035,la Comisión que él designe. Además de entre sus miembros se elegirá un secretario quien será el encargado de levantar las actas. La Comisión Curricula…
3,4,documento_fuente.pdf,11,39,0.630857,"e realizara al menos una vez en el año académico. El Vicedecano de cada Facultad, será el responsable del cumplimento de esta disposición. *** CONSEJO…"
4,5,documento_fuente.pdf,43,150,0.618987,"nombrados por la Junta Directiva de la respectiva Facultad y durarán en sus funciones dos años. En el cumplimiento de sus atribuciones, estarán facult…"
5,6,documento_fuente.pdf,29,99,0.610092,ones en Sistema Mecanizado estipulado en el Calendario de Actividades Académicas Administrativas.*** Artículo 105. Cuando por razones de fuerza mayor …
6,7,documento_fuente.pdf,6,21,0.609644,"udiciales fuera de la universidad, en los últimos cinco años, anteriores a su nombramiento. ATRIBUCIONES Y DEBERES DE LA ADMINISTRACIÓN ACADÉMICA CENT…"
7,8,documento_fuente.pdf,12,42,0.608434,"tración Académica de Facultad, a efecto de mejora; l) Conocer, asesorar y participar en los procesos referentes al Ingreso Universitario; m) Colaborar…"
8,9,documento_fuente.pdf,43,149,0.600573,de estudio de las Unidades de Aprendizaje a comparar en un 80 %. Las equivalencias se otorgaran siempre y cuando acrediten un grado académico de conf…
9,10,documento_fuente.pdf,28,98,0.599262,"ión la ejecutará automáticamente la Administración Académica de la Facultad, de acuerdo a lo aprobado por el Consejo Superior Universitario. CAPÍTULO …"


In [17]:
rank_uno = recuperados_mono[0]
display(
    Markdown(
        "**Fragmento con mayor similitud de recuperación**  \n"
        f"Documento: `{rank_uno['documento']}` · "
        f"Página: {rank_uno['pagina']} · Chunk: {rank_uno['chunk_id']}\n\n"
        f"> {rank_uno['texto'][:700].replace(chr(10), ' ')}…"
    )
)

**Fragmento con mayor similitud de recuperación**  
Documento: `documento_fuente.pdf` · Página: 36 · Chunk: 125

> s administraciones Académicas Locales enviaran a la Administración Académica Central los colectores de las evaluaciones realizadas por los estudiantes a más tardar dentro de la tercera semana después de finalizado el ciclo respectivo, para el archivo correspondiente. Articulo 142. La Junta Directiva de Facultad, aprobará el calendario general de las evaluaciones, de acuerdo al calendario de actividades académicas–administrativas aprobado por el Consejo Superior Universitario. La Dirección o Jefatura responsable informará a la población interesada sobre el calendario antes mencionado por todos los medios a su alcance y no podrá ser modificado arbitrariamente. Artículo 143. Las actividades de …

## 15. Question Answering sobre los fragmentos

BETO-SQAC recibe la misma pregunta junto con cada chunk Top-10 y extrae un span por candidato. La función de `src` conserva tanto la similitud de recuperación como el score QA.

In [18]:
resultado_mono = rag_multidocumento(
    pregunta_demo,
    modelo_embeddings,
    indice_mono,
    metadata_mono,
    pipeline_qa,
    top_k=TOP_K,
)

In [19]:
tabla_candidatos_qa = pd.DataFrame(
    [
        {
            "Rank recuperación": candidato["rank"],
            "Documento": candidato["documento"],
            "Página": candidato["pagina"],
            "Chunk": candidato["chunk_id"],
            "Similitud de recuperación": candidato["similitud"],
            "Respuesta candidata": candidato["answer"],
            "Score QA": candidato["score"],
        }
        for candidato in resultado_mono["candidatos_qa"]
    ]
)
display(
    tabla_candidatos_qa.style.format(
        {"Similitud de recuperación": "{:.6f}", "Score QA": "{:.6f}"}
    )
)

,Rank recuperación,Documento,Página,Chunk,Similitud de recuperación,Respuesta candidata,Score QA
0,1,documento_fuente.pdf,36,125,0.682253,"Articulo 142. La Junta Directiva de Facultad, aprobará el calendario general de las evaluaciones, de acuerdo al calendario de actividades académicas–administrativas aprobado por el Consejo Superior Universitario",0.021647
1,2,documento_fuente.pdf,2,6,0.639237,una vez al mes,0.983679
2,3,documento_fuente.pdf,12,44,0.637035,cada mes,0.720997
3,4,documento_fuente.pdf,11,39,0.630857,cada mes,0.921885
4,5,documento_fuente.pdf,43,150,0.618987,dos años,0.893072
5,6,documento_fuente.pdf,29,99,0.610092,***,0.108471
6,7,documento_fuente.pdf,6,21,0.609644,en los últimos cinco años,0.065631
7,8,documento_fuente.pdf,12,42,0.608434,tración Académica de Facultad,0.005488
8,9,documento_fuente.pdf,43,149,0.600573,Articulo 177,0.010683
9,10,documento_fuente.pdf,28,98,0.599262,20 semanas,0.542185


## 16. Selección de la mejor respuesta

El ganador es el candidato con mayor **score QA**, no necesariamente el chunk situado en el rank 1 de FAISS. La similitud controla recuperación; el score QA controla la selección del span.

In [20]:
display(
    pd.DataFrame(
        [
            {
                "Rank del chunk ganador": resultado_mono["rank_fuente"],
                "Similitud de recuperación": resultado_mono[
                    "similitud_recuperacion"
                ],
                "Score QA": resultado_mono["score_qa"],
            }
        ]
    ).style.format(
        {"Similitud de recuperación": "{:.6f}", "Score QA": "{:.6f}"}
    )
)

,Rank del chunk ganador,Similitud de recuperación,Score QA
0,2,0.639237,0.983679


## 17. Respuesta con trazabilidad

La salida final reúne la respuesta y los dos scores con documento, página y chunk. Esto permite revisar la evidencia original.

In [21]:
def mostrar_respuesta(resultado):
    """Muestra una respuesta RAG sin confundir recuperación con QA."""
    display(
        Markdown(
            f"**Pregunta:** {resultado['pregunta']}\n\n"
            f"**Respuesta:** {resultado['respuesta']}\n\n"
            f"**Score QA:** {resultado['score_qa']:.6f}\n\n"
            f"**Similitud de recuperación:** "
            f"{resultado['similitud_recuperacion']:.6f}\n\n"
            f"**Fuente:** `{resultado['documento_fuente']}`\n\n"
            f"**Página:** {resultado['pagina_fuente']}  \n"
            f"**Chunk:** {resultado['chunk_fuente']}  \n"
            f"**Rank de recuperación:** {resultado['rank_fuente']}"
        )
    )

mostrar_respuesta(resultado_mono)

**Pregunta:** ¿Con qué periodicidad sesiona ordinariamente el Consejo Académico?

**Respuesta:** una vez al mes

**Score QA:** 0.983679

**Similitud de recuperación:** 0.639237

**Fuente:** `documento_fuente.pdf`

**Página:** 2  
**Chunk:** 6  
**Rank de recuperación:** 2

## 18. Extensión multidocumento

La extensión conserva cada PDF de forma independiente. Para que la defensa sea ágil, esta sección carga el índice y la metadata ya producidos por el experimento validado; no vuelve a vectorizar los 765 chunks ni modifica `results/`.

El flujo de consulta es el mismo, pero ahora la procedencia documental forma parte de la respuesta.

In [22]:
rutas_pdf = [
    PDF_PRINCIPAL,
    *sorted((RAIZ / "data" / "corpus_complementario").glob("*.pdf")),
]
assert len(rutas_pdf) == 9
assert len({ruta.name for ruta in rutas_pdf}) == 9

ruta_metadata_multi = RAIZ / "results" / "rag_multidocumento_metadata.jsonl"
ruta_indice_multi = RAIZ / "results" / "indice_rag_multidocumento.faiss"

metadata_multi = [
    json.loads(linea)
    for linea in ruta_metadata_multi.read_text(encoding="utf-8").splitlines()
]
for chunk in metadata_multi:
    chunk.setdefault("archivo", chunk["documento"])
indice_multi = faiss.read_index(str(ruta_indice_multi))

campos_trazabilidad = {"documento", "archivo", "pagina", "chunk_id", "texto"}
validar_metadata_chunks(metadata_multi)
assert all(campos_trazabilidad <= chunk.keys() for chunk in metadata_multi)
assert len(metadata_multi) == 765
assert indice_multi.ntotal == len(metadata_multi)
assert indice_multi.d == 768

In [23]:
resumen_corpus = pd.read_csv(
    RAIZ / "results" / "rag_multidocumento_corpus.csv"
)
display(resumen_corpus[["documento", "paginas", "chunks"]])

print(f"PDF independientes: {len(rutas_pdf)}")
print(f"Vectores indexados: {indice_multi.ntotal}")
print(f"Dimensión: {indice_multi.d}")

,documento,paginas,chunks
0,documento_fuente.pdf,62,217
1,ley_organica_ues.pdf,21,96
2,reglamento_arancel_academico_ues.pdf,4,9
3,reglamento_becas_ues.pdf,15,63
4,reglamento_disciplinario_ues.pdf,11,51
5,reglamento_electoral_ues.pdf,26,81
6,reglamento_general_ley_organica_ues.pdf,25,112
7,reglamento_sistema_escalafon_personal_ues.pdf,29,114
8,reglamento_unidades_valorativas_cum_ues.pdf,6,22
9,TOTAL,199,765


PDF independientes: 9
Vectores indexados: 765
Dimensión: 768


In [24]:
preguntas_por_numero = {
    item["numero"]: item for item in preguntas_existentes
}
casos_defensa = [
    ("Caso 1 · respuesta correcta", 4),
    ("Caso 2 · documento correcto, chunk incorrecto", 2),
    ("Caso 3 · recuperación correcta, QA incorrecto", 16),
    ("Pregunta sin respuesta · limitación de abstención", 17),
]

assert all(numero in preguntas_por_numero for _, numero in casos_defensa)
display(
    pd.DataFrame(
        [
            {"Caso": caso, "N.º experimental": numero,
             "Pregunta": preguntas_por_numero[numero]["pregunta"]}
            for caso, numero in casos_defensa
        ]
    )
)

,Caso,N.º experimental,Pregunta
0,Caso 1 · respuesta correcta,4,¿En qué ámbitos goza de autonomía la Universid...
1,"Caso 2 · documento correcto, chunk incorrecto",2,¿Cuál es el máximo de veces que un estudiante ...
2,"Caso 3 · recuperación correcta, QA incorrecto",16,¿A cuántas horas de trabajo del estudiante equ...
3,Pregunta sin respuesta · limitación de abstención,17,¿Cuál es la contraseña vigente de la red Wi-Fi...


## 19. Demostración interactiva

`demostrar_rag("pregunta")` reutiliza el índice multidocumento y presenta solo la información necesaria para la defensa: Top-10, respuesta, score QA, similitud y fuente. La ejecución guardada usa preguntas 4, 2, 16 y 17 del conjunto existente para mostrar éxito, dos fallos distintos y la limitación de abstención.

La pregunta 17 no tiene respuesta en el corpus. El sistema actual fuerza una extracción porque no posee umbral ni clasificador de abstención; los artefactos experimentales registran **0/2 abstenciones correctas (0 %)**. Esta fase documenta el comportamiento y no lo corrige.

In [25]:
def demostrar_rag(pregunta):
    """Ejecuta una consulta y muestra una salida compacta y trazable."""
    resultado = rag_multidocumento(
        pregunta, modelo_embeddings, indice_multi, metadata_multi,
        pipeline_qa, top_k=TOP_K,
    )
    display(Markdown(f"### PREGUNTA\n\n{pregunta}"))
    display(Markdown("### TOP-K RECUPERADOS"))
    display(
        tabla_recuperacion(resultado["recuperados"]).style.format(
            {"Similitud de recuperación": "{:.6f}"}
        )
    )
    display(Markdown(
        "### RESPUESTA\n\n"
        f"**Respuesta:** {resultado['respuesta']}  \n"
        f"**Score QA:** {resultado['score_qa']:.6f}  \n"
        f"**Similitud:** {resultado['similitud_recuperacion']:.6f}  \n"
        f"**Documento:** `{resultado['documento_fuente']}`  \n"
        f"**Página:** {resultado['pagina_fuente']}  \n"
        f"**Chunk:** {resultado['chunk_fuente']}"
    ))
    return resultado

In [26]:
resultados_defensa = {}
for caso, numero in casos_defensa:
    item = preguntas_por_numero[numero]
    display(Markdown(f"## {caso}"))
    resultados_defensa[numero] = demostrar_rag(item["pregunta"])

## Caso 1 · respuesta correcta

### PREGUNTA

¿En qué ámbitos goza de autonomía la Universidad para cumplir sus fines?

### TOP-K RECUPERADOS

,Rank,Documento,Página,Chunk,Similitud de recuperación,Muestra
0,1,ley_organica_ues.pdf,4,16,0.845400,una de las Facultades gozará de autonomía administrativa y técnica; contará con un presupuesto para la consecución de sus fines y estará obligada a re…
1,2,ley_organica_ues.pdf,2,6,0.778563,"omentar entre sus educandos el ideal de unidad de los pueblos centroamericanos. Para la mejor realización de sus fines, la Universidad podrá establece…"
2,3,documento_fuente.pdf,7,23,0.768866,ramación del Sistema Académico Administrativo de la Universidad; y k) Las demás atribuciones de índole académico administrativo que le asigne la Secre…
3,4,reglamento_general_ley_organica_ues.pdf,3,9,0.766539,da Del Consejo Superior Universitario Consejo Superior Universitario Art. 9. - El Consejo Superior Universitario es el órgano colegiado con jerarquía …
4,5,ley_organica_ues.pdf,13,61,0.761224,"iantes que se constituyan en la Universidad o en las Facultades, serán totalmente independientes de las autoridades de la Universidad y de las Faculta…"
5,6,ley_organica_ues.pdf,5,25,0.757418,al Universitaria tendrá las siguientes atribuciones y deberes: 81 a) Aprobar o reformar su reglamento interno; b) Acordar las propuestas de reforma a…
6,7,ley_organica_ues.pdf,11,54,0.756729,tadas por los distintos órganos de la Universidad; d) Proponer a los órganos de la Universidad las medidas legales sobre administración y operatividad…
7,8,ley_organica_ues.pdf,19,87,0.756631,"izar los fines de la Universidad; f) La aprobación, revisión o modificación de carreras, planes y programas de estudio; g) El otorgamiento de grados h…"
8,9,reglamento_general_ley_organica_ues.pdf,15,64,0.755300,"démica de la Universidad velará por la aplicación de las disposiciones básicas sobre procedimientos, medidas y resoluciones académicas, contenidas en …"
9,10,ley_organica_ues.pdf,14,67,0.754672,"spondiente. La Universidad podrá otorgar equivalencias de estudio a los realizados en otras instituciones de educación superior, que formen parte del …"


### RESPUESTA

**Respuesta:** en lo docente, lo administrativo y lo económico  
**Score QA:** 0.928334  
**Similitud:** 0.778563  
**Documento:** `ley_organica_ues.pdf`  
**Página:** 2  
**Chunk:** 6

...

## Pregunta sin respuesta · limitación de abstención

### PREGUNTA

¿Cuál es la contraseña vigente de la red Wi-Fi del campus central?

### TOP-K RECUPERADOS

,Rank,Documento,Página,Chunk,Similitud de recuperación,Muestra
0,1,documento_fuente.pdf,41,142,0.290441,"Académica de destino, notificará al estudiante y a la Administración Académica de procedencia. Ésta última deberá enviar copia del expediente del estu…"
1,2,reglamento_electoral_ues.pdf,7,20,0.287479,informativas o cualquier otro medio de comunicación. Obligación de la Administración Académica Art. 13. La Administración Académica de cada Facultad a…
2,3,documento_fuente.pdf,32,109,0.286899,"IONADA. Artículo 121. El Vicedecanato de Facultad, podrá autorizar la inscripción condicionada de un estudiante, cuando los motivos para no realizarla…"
3,4,documento_fuente.pdf,23,79,0.273706,iente y que mantiene inscripción de Unidades de Aprendizaje en cada ciclo. Asimismo el egresado que se encuentra matriculado y ha inscrito su trabajo …
4,5,reglamento_general_ley_organica_ues.pdf,21,90,0.273402,sta por seis meses; para que el mismo adquiera carácter permanente se requerirá de acuerdo del CSU. Asuetos y vacaciones Art. 86. - Lo referente a asu…
5,6,reglamento_sistema_escalafon_personal_ues.pdf,26,100,0.259882,"ento del plan anual de capacitación de su personal, dentro del sistema institucional regulado en el presente artículo. Garantía de participación Art. …"
6,7,reglamento_unidades_valorativas_cum_ues.pdf,5,17,0.258167,"ama especial de refuerzo académico, genera el derecho a obtener de inmediato la declaratoria de egresado. Emisión de Certificaciones o Constancias de …"
7,8,reglamento_sistema_escalafon_personal_ues.pdf,11,50,0.257082,"I, II, III y IV. Registro Escalafonario Art. 36. - El Registro Escalafonario de la Universidad, estará a cargo de la Unidad de Recursos Humanos adscr…"
8,9,reglamento_sistema_escalafon_personal_ues.pdf,14,66,0.252958,"57 86 Los puntajes contenidos en ella son acumulables. Actualización del Registro Escalafonario Art. 46. - Los Comités de las diferentes Facultades, …"
9,10,ley_organica_ues.pdf,16,76,0.252448,"l Registro de la Propiedad Raíz correspondiente, quedando así, dicho inmueble, sujeto a las disposiciones del derecho común. Los bienes muebles que fo…"


### RESPUESTA

**Respuesta:** sta  
**Score QA:** 0.467128  
**Similitud:** 0.273402  
**Documento:** `reglamento_general_ley_organica_ues.pdf`  
**Página:** 21  
**Chunk:** 90

## 20. Resumen del flujo

```text
CARGAR → FRAGMENTAR → VECTORIZAR → INDEXAR
                                  ↓
RESPUESTA + FUENTE ← SELECCIONAR ← QA ← TOP-K ← PREGUNTA
```

- **MPNet** representa preguntas y chunks en el mismo espacio vectorial.
- **FAISS** recupera los chunks semánticamente próximos.
- **BETO-SQAC** extrae una respuesta de cada candidato.
- La selección final usa **score QA** y conserva **similitud de recuperación**.
- Documento, página y chunk permiten volver a la evidencia.

Responsabilidades de los notebooks:

- `laboratorio_ml_qa.ipynb`: desarrollo, experimentos y evidencia canónica.
- `RAG_UES_Demo.ipynb`: explicación y demostración del pipeline final.